# SimPandas Tutorial

SimPandas wraps `pandas.DataFrame` and `pandas.Series` with first-class **unit** support.
This notebook walks through every SimPandas-specific feature using two real data sources:

| Variable | Source |
|---|---|
| `sample_prod` | Excel file – two producers (`P1` field units, `P2` metric), weekly timesteps |
| `sample_smry` | Eclipse binary summary (`.SMSPEC` + `.UNSMRY`) |


## 1  Setup

In [ ]:
import simpandas as spd

print('simpandas', spd.__version__)

## 2  Loading data

Both readers return a `SimDataFrame` with unit metadata already attached.


In [ ]:
sample_prod = spd.read_excel('./test/_testing_data/sample_prod.xlsx')
sample_prod

In [ ]:
sample_smry = spd.read_summary('./test/_testing_data/SAMPLE_PROD.SMSPEC')
sample_smry

## 3  Units and metadata

Every `SimDataFrame` / `SimSeries` carries unit metadata alongside the data.


In [ ]:
# Both objects are SimDataFrame, not bare pandas DataFrames
print(type(sample_prod))
print(type(sample_smry))

In [ ]:
# .units returns a {column: unit_string} dict
sample_prod.units

In [ ]:
# get_units() for the Eclipse summary
sample_smry.get_units()

In [ ]:
# .wells lists the unique well names from column names (right of ':')
print('sample_prod wells:', sample_prod.wells)
print('sample_smry wells:', sample_smry.wells)

In [ ]:
# .meta carries reader-supplied metadata (e.g. DIMENS, STARTDAT from SMSPEC header)
sample_smry.meta

## 4  Setting a DatetimeIndex

Most time-series helpers require a `DatetimeIndex`.
`set_index` with `inplace=True` works exactly like pandas —
the result is still a `SimDataFrame` with units intact.


In [ ]:
sample_prod.set_index('DATE', inplace=True)
sample_prod

In [ ]:
# index.units is preserved after set_index
sample_prod.index.units

## 5  Smart column selection with `[]`

SimPandas's `__getitem__` goes beyond plain pandas:
it understands Eclipse-style `KEYWORD:WELLNAME` column names **and** can
select by date when the key looks like a date string.


In [ ]:
# Exact column name -> SimSeries (with the column unit)
single_column = sample_prod['WOPR:P1']
print(type(single_column))
single_column

In [ ]:
# Single name in a list -> 1-column SimDataFrame
sample_prod[['WOPR:P1']]

In [ ]:
# Multiple columns as a list
sample_prod[['WOPR:P1', 'WOPR:P2']]

In [ ]:
# Left part of 'KEYWORD:WELL' -> all columns with that keyword
sample_prod['WOPR']    # returns WOPR:P1 and WOPR:P2

In [ ]:
# Right part -> all measurements for that well
sample_prod['P1']    # returns WOPR:P1, WWPR:P1, WBHP:P1

In [ ]:
# Slice of column names (inclusive on both ends)
sample_prod['WWPR:P1':'WOPR:P2']

In [ ]:
# fnmatch-style wildcard pattern
sample_prod['W?PR*']    # any 4-char keyword starting with W

In [ ]:
# When key is not a column, [] looks in the DatetimeIndex
# Exact date -> SimSeries (one row of all columns)
sample_prod['2100-04-09']

In [ ]:
# Partial date string -> all rows within that month
sample_prod['2100-04']

In [ ]:
# Year only -> all rows in that year
sample_prod['2100']

In [ ]:
# Date range slice
sample_prod['2100-04-09':'2100-05-15']

### `[index, column]` shortcut

Combine a date/date-range with a column name or name-pattern in one `[]` call using a 2-tuple.


In [ ]:
sample_prod['2100-02-26', 'WBHP:P1']

In [ ]:
# Date range + right-part well filter
sample_prod['2100-04':'2100-06', 'P1']

### `SimSeries[]` understands date strings too

In [ ]:
single_column['2100-05']    # all values in May 2100

In [ ]:
single_column['2100-04-30':'2100-06-12']

## 6  `.loc` and `.iloc`

SimPandas wraps both indexers so results always come back as `SimDataFrame` / `SimSeries`
with unit metadata preserved.  All pandas-compatible date-string shortcuts work.


In [ ]:
sample_prod.loc['2100-02-26']        # one row by exact date -> SimSeries

In [ ]:
sample_prod.loc['2100-01']           # all rows in January 2100

In [ ]:
sample_prod.loc['2100']              # full year 2100

In [ ]:
# Date range + right-part column filter
sample_prod.loc['2100-12-24':'2101-01-15', 'P1']

In [ ]:
# iloc still returns SimDataFrame / SimSeries
print(type(sample_prod.iloc[0:5]))
sample_prod.iloc[0:5, 0:3]

## 7  Time aggregations

`.daily()`, `.monthly()`, and `.yearly()` resample a `DatetimeIndex` frame
and return a new `SimDataFrame` with units intact.


In [ ]:
# Mean per calendar day, DatetimeIndex (default)
sample_prod.daily()

In [ ]:
# Integer index instead of DatetimeIndex
sample_prod.daily(datetime_index=False)

In [ ]:
# complete_index=True fills gaps with NaN rows for missing days
sample_prod.daily(complete_index=True)

In [ ]:
sample_prod.monthly()

In [ ]:
# complete_index fills months without data; ffill propagates last known value
sample_prod.monthly(complete_index=True, fillna_method='ffill')

In [ ]:
# Report on the first day of each month
sample_prod.monthly(day='first')

In [ ]:
sample_prod.yearly()

## 8  Unit-aware arithmetic

Operators `+`, `-`, `*`, `/`, `**` all track units.
When two operands have **convertible** units (e.g. `stb/day` <-> `sm3/day`),
SimPandas converts the right-hand side to the left-hand side's unit automatically.


In [ ]:
# P1 columns: rates in stb/day (field units)
# P2 columns: rates in sm3/day and barsa (metric units)
print('P1 units:', sample_prod['P1'].units)
print('P2 units:', sample_prod['P2'].units)

In [ ]:
# sum(axis=1): sum across columns (wells) for each timestep
# P2 auto-converted to stb/day to match P1 (left operand)
total_oil_rate = sample_prod['WOPR'].sum(axis=1)
print('result unit:', total_oil_rate.get_units())
total_oil_rate

In [ ]:
# Direct addition: P1 (stb/day) + P2 (sm3/day)
p1_plus_p2 = sample_prod['WOPR:P1'] + sample_prod['WOPR:P2']
print('result unit:', p1_plus_p2.get_units())
p1_plus_p2

In [ ]:
# Reversed operands: result unit is now sm3/day
p2_plus_p1 = sample_prod['WOPR:P2'] + sample_prod['WOPR:P1']
print('result unit:', p2_plus_p1.get_units())
p2_plus_p1

## 9  Aggregation methods

All scalar/per-column aggregation results carry the correct unit.
`rms()` (root-mean-square) is a SimPandas extension not in standard pandas.


In [ ]:
s = sample_prod['WOPR:P1']   # SimSeries - unit = stb/day

print('sum  :', s.sum())
print('mean :', s.mean())
print('std  :', s.std())
print('var  :', s.var())
print('rms  :', s.rms())    # root-mean-square -- SimPandas extension
print('min  :', s.min())
print('max  :', s.max())

In [ ]:
# describe() keeps unit labels in the output
sample_prod.describe()

## 10  Unit conversion with `.to()`

Convert any `SimDataFrame` or `SimSeries` to a target unit in one call.


In [ ]:
# SimSeries: stb/day -> sm3/day
p1_metric = sample_prod['WOPR:P1'].to('sm3/day')
print('original :', sample_prod['WOPR:P1'].get_units())
print('converted:', p1_metric.get_units())
p1_metric

In [ ]:
# SimDataFrame: convert every convertible column at once
sample_prod[['WBHP:P1', 'WBHP:P2']].to('barsa')

## 11  Equality comparisons with precision

`.eq(other, precision=N)` rounds each value to `N` significant figures before comparing.
Useful when two results share the same physical meaning but were computed via
different unit conversion paths.


In [ ]:
p1_p2 = sample_prod['WOPR:P1'] + sample_prod['WOPR:P2']   # result in stb/day
p2_p1 = sample_prod['WOPR:P2'] + sample_prod['WOPR:P1']   # result in sm3/day

# Exact == may fail due to floating-point round-trip
print('exact equal     :', (p1_p2 == p2_p1).all())
# .eq() rounds to N significant figures before comparing
print('equal at 6 s.f. :', p1_p2.eq(p2_p1, precision=6).all())

## 12  `SimSeries` with per-index units

A `SimSeries` can hold a **different unit for each element** by passing a `{label: unit}` dict.
This is ideal for parameter tables where each row has a distinct physical quantity.


In [ ]:
params = spd.SimSeries(
    data=[1000, 0.25, 300, 365],
    index=['depth', 'porosity', 'pressure', 'time'],
    units={'depth': 'm', 'porosity': 'fraction', 'pressure': 'barsa', 'time': 'day'},
    name='reservoir_params'
)
print('units dict:', params.units)
params

In [ ]:
# Retrieve the unit for a specific index label
print('depth unit   :', params.get_units('depth'))
print('pressure unit:', params.get_units('pressure'))

## 13  Unit-aware column assignment

Assigning a `(value, unit)` tuple to a column tells SimPandas the value's unit.
If the column already exists, the value is converted to the column's existing unit.
If the column is new, the unit is registered automatically.


In [ ]:
test = sample_prod[['WOPR:P1', 'WWPR:P1']].copy()

# Assign a new column with an explicit unit
test['WOPR:P3'] = 500, 'sm3/day'
print('WOPR:P3 unit:', test.get_units('WOPR:P3'))
test.head(3)

In [ ]:
# Overwrite an existing column -- value auto-converted to existing column unit
test['WOPR:P1'] = 1000, 'sm3/day'   # stored as stb/day equivalent
print('WOPR:P1 unit still:', test.get_units('WOPR:P1'))
test['WOPR:P1'].head(3)

## 14  Export

All standard pandas writers are available on `SimDataFrame` / `SimSeries`,
with SimPandas wrappers that embed unit metadata in the output files.


In [ ]:
print('Writer methods on SimDataFrame:')
print([m for m in dir(sample_prod) if m.startswith('to_')])

In [ ]:
# Uncomment any line below to actually write

# Eclipse binary summary (round-trip)
# sample_prod.to_summary('./output/ROUND_TRIP.SMSPEC')
# reloaded = spd.read_summary('./output/ROUND_TRIP.SMSPEC')

# CSV with units row
# sample_prod.to_csv('./output/sample_prod.csv')        # row 0 contains units
# df_back = spd.read_csv('./output/sample_prod.csv', units=0)

# JSON with units envelope
# sample_prod.to_json('./output/sample_prod.json')

# HDF5
# sample_prod.to_hdf5('./output/sample_prod.h5')

# Excel
# sample_prod.to_excel('./output/sample_prod.xlsx')

print('All writers preserve unit metadata for round-tripping.')